In [ ]:
import time
import random
import re
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
import spacy

In [ ]:
nlp = spacy.load("de_core_news_md")
FAZ_API = "https://www.faz.net/api/faz-content-search"
TARGET_URLS = 1000
MAX_PAGES_PER_KEYWORD = 20
MIN_YEAR = 2022
OUTPUT_FILE = "faz_scrapped.csv"
INVALID_URL_PATTERNS = ["/video/","/podcast/","/audio/","/bilder/", "/einspruch/","/interaktiv/","/explainer/"]
CLIMATE_KEYWORDS = ["klima", "klimawandel", "klimakrise", "erderwärmung", "globale erwärmung", "treibhauseffekt", "treibhausgas",
    "co2", "kohlendioxid", "emission", "emissionen", "klimaschutz", "klimapolitik", "klimaneutral", "dekarbonisierung",
    
    "klimaziel", "klimaziele", "co2-preis", "emissionshandel", "klimaschutzgesetz", "paris-abkommen", "pariser abkommen",
    "eu-klimapaket", "green deal", "klimaplan", "klimabericht", "ipcc",

    "energiewende", "erneuerbare energien", "windkraft", "solarenergie", "photovoltaik", "wasserstoff",
    "kohlekraft", "kohleausstieg", "atomkraft", "stromnetz", "netzausbau", "e-mobilität", "verbrennerverbot", "industrieemissionen",
    
    "hitzewelle", "dürre", "hochwasser", "überschwemmung", "starkregen", "waldbrand", "extremwetter",
    "gletscherschmelze", "meeresspiegel", "wasserknappheit", "klimafolgen", "artensterben",

    "klimarisiko", "nachhaltigkeit", "nachhaltige investitionen", "esg", "grüne finanzierung", "klimafonds",
    "energiepreise", "strompreis", "gaspreis",

    "landwirtschaft", "trockenheit", "ernteschäden", "umweltschutz", "biodiversität", "naturschutz", "ökosystem", "umweltpolitik",

    "fridays for future", "klimaaktivisten", "klimaprotest", "letzte generation", "klimabewegung", "klimadebatte"]
STRICT_KEYWORDS = ["klima", "klimakrise", "klimawandel", "erderwärmung", "co2", "kohlendioxid", "emission", "emissionen",
    "energiewende", "klimaschutz", "klimaneutral", "treibhauseffekt", "treibhausgas", "hitzewelle", "dürre", "hochwasser",
    "wasserknappheit", "starkregen", "waldbrand"]
CLASSIFICATION_KEYWORDS = {
    "Climate_Policy": ["gesetz", "politik", "regierung", "beschluss", "verordnung","ziel", "klimaziel", "bundestag", "eu", "parlament", "ministerium"],
    "Climate_Science": ["studie", "forschung", "wissenschaft", "ipcc", "daten", "analyse", "bericht", "modell", "forscher"],
    "Energy_Transition": ["energiewende", "erneuerbar", "solar", "windkraft", "kohlekraft", "atomkraft", "wasserstoff", "stromnetz"],
    "Climate_Economy": ["kosten", "industrie", "wirtschaft", "markt", "unternehmen", "investition", "preis", "arbeitsplätze"],
    "Climate_Activism": ["protest", "demonstration", "aktivisten", "fridays for future", "ngo", "bewegung"],
    "Climate_Impact": ["hitzewelle", "dürre", "hochwasser", "überschwemmung", "starkregen", "waldbrand", "extremwetter"],
    "Climate_Geopolitics": ["china", "usa", "eu", "russland", "international", "global", "weltweit", "g7", "g20"],
    "Climate_Opinion": ["meinung", "kommentar", "kolumne", "leitartikel", "debatte", "gastbeitrag"]}
API_HEADERS = {
"User-Agent":"Mozilla/5.0",
"Accept":"application/json",
"Referer":"https://www.faz.net/suche/"
}

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--disable-notifications")
chrome_options.add_argument("--disable-geolocation")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),options=chrome_options)
wait = WebDriverWait(driver,25)

In [ ]:
def extract_actors(text):
    doc = nlp(text[:3000])
    return list(set(ent.text for ent in doc.ents if ent.label_ in ("PER","ORG")))
def sentence_count(text):
    return text.count(".")
def human_sleep(a=2,b=4):
    time.sleep(random.uniform(a,b))
def safe_get(url):
    try:
        driver.set_page_load_timeout(25)
        driver.get(url)
        current = driver.current_url.lower()
        if "consent.faz.net" in current:
            return False
        return True
    except TimeoutException:
        return False
def is_valid_year(doc):
    try:
        return int(doc["date"][:4]) >= MIN_YEAR
    except:
        return False
def is_climate_article(title,content):
    text = f"{title} {content}".lower()
    hits = sum(text.count(k) for k in STRICT_KEYWORDS)
    return hits >= 2
def classify_article(title,content):
    text = f"{title} {content}".lower()
    scores = {k:sum(text.count(w) for w in v) for k,v in CLASSIFICATION_KEYWORDS.items()}
    best = max(scores,key=scores.get)
    return best if scores[best] > 0 else "Other_Climate"
def extract_article():
    paras = driver.find_elements(By.CSS_SELECTOR,"article p")
    clean = []
    for p in paras:
        t = p.text.strip()
        if len(t) < 60:
            continue
        if "anzeige" in t.lower():
            continue
        clean.append(t)
    if len(clean) < 5:
        return None,None
    return clean[0]," ".join(clean)
url_to_meta = {}
session = requests.Session()
for kw in CLIMATE_KEYWORDS:
    print("Keyword:",kw)
    for page in range(1,MAX_PAGES_PER_KEYWORD+1):
        params = {
        "q":kw,
        "page":page,
        "rows":20,
        "paid_content":"include",
        "sort_by":"date",
        "sort_order":"desc"}
        r = session.get(FAZ_API, headers=API_HEADERS, params=params)
        docs = r.json().get("docs",[])
        if not docs:
            break
        for d in docs:
            if len(url_to_meta) >= TARGET_URLS:
                break
            if not is_valid_year(d):
                continue
            url = d.get("url")
            if not url:
                continue
            if any(b in url for b in INVALID_URL_PATTERNS):
                continue
            url_to_meta.setdefault(url,{"keyword":kw, "api_doc":d})
        if len(url_to_meta) >= TARGET_URLS:
            break
        human_sleep(1,2)
    if len(url_to_meta) >= TARGET_URLS:
        break

In [ ]:
urls = list(url_to_meta.keys())
rows = []
print("Scraping articles")

In [ ]:
for idx,url in enumerate(urls,1):
    meta = url_to_meta[url]
    try:
        if not safe_get(url):
            continue
        human_sleep(2,4)
        if "consent" in driver.current_url:
            continue
        wait.until(EC.presence_of_element_located((By.TAG_NAME,"p")))
        headline = None
        try:
            headline = driver.find_element(By.TAG_NAME,"h1").text.strip()
        except:
            try:
                headline = driver.find_element(By.CSS_SELECTOR,".atc-Headline").text.strip()
            except:
                continue
        intro,content = extract_article()
        if not content:
            continue
        if not is_climate_article(headline,content):
            continue
        actors = extract_actors(content)
        rows.append({
        "URL":url,
        "Source":"FAZ",
        "Language": "de"
        "Published_Date":meta["api_doc"]["date"],
        "Article_Classification":classify_article(headline,content),
        "Headline":headline,
        "Intro":intro,
        "Content":content,
        "Content_Length":len(content),
        "Sentence_Count":sentence_count(content),
        "Actors":", ".join(actors),
        "Actor_Count":len(actors)
        })
        print(f"[{idx}/{len(urls)}] saved")
        if idx % 15 == 0:
            human_sleep(10,15)
    except Exception as e:
        print("URL not accessible",url)
        continue

In [ ]:
df = pd.DataFrame(rows)
df.drop_duplicates(subset=["URL"],inplace=True)
df.to_csv(OUTPUT_FILE,index=False,encoding="utf-8-sig")
driver.quit()
print(f"File saved: {OUTPUT_FILE}")
print("Total relevant FAZ climate articles:",len(df))